In [60]:
import java.time.LocalTime
import java.time.format.DateTimeFormatter
import kotlin.text.MatchResult
import kotlin.text.get

// ========== Domain model ==========
sealed class LogEvent {
    data class ConfirmationRuleUpdate(
        val headSlot: Long,
        val headShortRootHex: String,           // e.g., 0x9acceb
        val confirmedSlot: Long,
        val confirmedShortRootHex: String,      // e.g., 0x6dba83
    ) : LogEvent()

    data class SlotEvent(
        val time: LocalTime,
        val slot: Long,
        val blockRootHex: String?,              // null when "... empty"
    ) : LogEvent()

//    data class LateBlockImport(
//        val time: LocalTime,
//        val level: String,                      // INFO/WARN/etc.
//        val blockRootHex: String,
//        val slot: Long,
//        val proposer: Long,
//        val result: String,
//        val timingsMs: LinkedHashMap<String, Long> // keeps order from log
//    ) : LogEvent()
//
//    data class ReorgEvent(
//        val time: LocalTime,
//        val level: String,
//        val newHeadRootHex: String,
//        val newHeadSlot: Long,
//        val prevHeadRootHex: String,
//        val prevHeadSlot: Long,
//        val commonAncestorRootHex: String,
//        val commonAncestorSlot: Long
//    ) : LogEvent()

    data class Unknown(val line: String) : LogEvent()
}

// ========== Parser ==========
object TekuLogParser {
    private val timeFmt = DateTimeFormatter.ofPattern("HH:mm:ss.SSS")

    // 1) updateConfirmationRuleStore
    private val reUpdate = Regex(
        """
        ^updateConfirmationRuleStore:\s*
        head=(?<head>\d+),\(
            (?<headHex>0x[0-9a-fA-F]+)
        \),\s*
        confirmed=(?<conf>\d+)\(
            (?<delta>-?\d+)
        \),\(
            (?<confHex>0x[0-9a-fA-F]+)
        \),\s*
        states\s+requested/uniq:\s*
            (?<req>\d+)/
            (?<uniq>\d+),\s*
        justified=Checkpoint\[
            (?<jEpoch>\d+),\s*
            (?<jHex>0x[0-9a-fA-F]+)
        \]\s*in\s*
            (?<ms>\d+)\s*ms
        $
        """.trimIndent().compactWs(),
        setOf()
    )

    // 2a) Full Slot Event
    private val reSlotFull = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Slot\s+Event\s+\*{3}\s+
        Slot:\s*(?<slot>\d+),\s*
        Block:\s*(?<block>[0-9a-fA-F]{64}),\s*
        Justified:\s*(?<j>\d+),\s*
        Finalized:\s*(?<f>\d+),\s*
        Peers:\s*(?<peers>\d+)
        $
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    // 2b) Short/empty Slot Event (… empty and the rest may be ellipsis)
    private val reSlotEmpty = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Slot\s+Event\s+\*{3}\s+
        Slot:\s*(?<slot>\d+),\s*
        Block:\s*\.{3}\s*empty
        (?:,.*)?$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    // 3) Epoch Event
    private val reEpoch = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Epoch\s+Event\s+\*{3}\s+
        Epoch:\s*(?<epoch>\d+),\s*
        Justified\s+checkpoint:\s*(?<j>\d+),\s*
        Finalized\s+checkpoint:\s*(?<f>\d+),\s*
        Finalized\s+root:\s*(?<root>[0-9a-fA-F]{64})
        $
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    // 4) Late Block Import
    // Example:
    // 18:23:07.855 WARN  - Late Block Import *** Block: <64hex> (<slot>) Proposer: 3126 Result: success
    // Timings: key 8760ms, key +0ms, key +85ms, ...
    private val reLateImport = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Late\s+Block\s+Import\s+\*{3}\s+
        Block:\s*(?<root>[0-9a-fA-F]{64})\s*
        \(\s*(?<slot>\d+)\s*\)\s*
        Proposer:\s*(?<prop>\d+)\s*
        Result:\s*(?<res>\S+)\s*
        Timings:\s*(?<timings>.+)$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    // 5) Reorg Event
    private val reReorg = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Reorg\s+Event\s+\*{3}\s+
        New\s+Head:\s*(?<new>[0-9a-fA-F]{64})\s*\(\s*(?<newSlot>\d+)\s*\),\s*
        Previous\s+Head:\s*(?<prev>[0-9a-fA-F]{64})\s*\(\s*(?<prevSlot>\d+)\s*\),\s*
        Common\s+Ancestor:\s*(?<anc>[0-9a-fA-F]{64})\s*\(\s*(?<ancSlot>\d+)\s*\)
        $
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    fun parse(line: String): LogEvent {
        reUpdate.matchEntire(line)?.let { m ->
            return LogEvent.ConfirmationRuleUpdate(
                headSlot = m.group("head").toLong(),
                headShortRootHex = m.group("headHex"),
                confirmedSlot = m.group("conf").toLong(),
                confirmedShortRootHex = m.group("confHex"),
            )
        }

        reSlotFull.matchEntire(line)?.let { m ->
            return LogEvent.SlotEvent(
                time = LocalTime.parse(m.group("time"), timeFmt),
                slot = m.group("slot").toLong(),
                blockRootHex = m.group("block")
            )
        }

        reSlotEmpty.matchEntire(line)?.let { m ->
            return LogEvent.SlotEvent(
                time = LocalTime.parse(m.group("time"), timeFmt),
                slot = m.group("slot").toLong(),
                blockRootHex = null,            // ... empty
            )
        }
//
//        reEpoch.matchEntire(line)?.let { m ->
//            return LogEvent.EpochEvent(
//                time = LocalTime.parse(m.group("time"), timeFmt),
//                epoch = m.group("epoch").toLong(),
//                justifiedCheckpoint = m.group("j").toLong(),
//                finalizedCheckpoint = m.group("f").toLong(),
//                finalizedRootHex = m.group("root")
//            )
//        }
//
//        reLateImport.matchEntire(line)?.let { m ->
//            val timings = parseTimings(m.group("timings"))
//            return LogEvent.LateBlockImport(
//                time = LocalTime.parse(m.group("time"), timeFmt),
//                level = m.group("lvl"),
//                blockRootHex = m.group("root"),
//                slot = m.group("slot").toLong(),
//                proposer = m.group("prop").toLong(),
//                result = m.group("res"),
//                timingsMs = timings
//            )
//        }
//
//        reReorg.matchEntire(line)?.let { m ->
//            return LogEvent.ReorgEvent(
//                time = LocalTime.parse(m.group("time"), timeFmt),
//                level = m.group("lvl"),
//                newHeadRootHex = m.group("new"),
//                newHeadSlot = m.group("newSlot").toLong(),
//                prevHeadRootHex = m.group("prev"),
//                prevHeadSlot = m.group("prevSlot").toLong(),
//                commonAncestorRootHex = m.group("anc"),
//                commonAncestorSlot = m.group("ancSlot").toLong()
//            )
//        }

        return LogEvent.Unknown(line)
    }

    // --- helpers ---

    private fun MatchGroupCollection.getByName(name: String) =
        (this as MatchNamedGroupCollection).get(name)

    private fun MatchResult.group(name: String): String =
        this.groups.getByName(name)?.value ?: error("Missing group '$name'")

    // Tolerate varied whitespace without making the regex unreadable
    private fun String.compactWs(): String =
        replace("\n", "").replace(Regex("\\s+"), "\\s*")

    // "arrival 8760ms, gossip_validation +0ms, processed +85ms, ..."
    private fun parseTimings(raw: String): LinkedHashMap<String, Long> {
        val map = LinkedHashMap<String, Long>()
        val pair = Regex("""([a-zA-Z0-9_]+)\s+([+\-]?\d+)ms""")
        for (part in raw.split(Regex("""\s*,\s*"""))) {
            val m = pair.find(part) ?: continue
            val key = m.groupValues[1]
            val v = m.groupValues[2].toLong()
            map[key] = v
        }
        return map
    }
}

// ========== Quick demo ==========
val lines1 = listOf(
    "updateConfirmationRuleStore: head=12657983,(0x9acceb), confirmed=12657982(-1),(0x6dba83), states requested/uniq: 5/2, justified=Checkpoint[395560, 0xe30323] in 598 ms",
    "17:17:03.812 INFO  - Slot Event  *** Slot: 12657983, Block: 9accebd69baa3e90411f123121f9eb7d2fc61f8117d81accf87f81c7bafde347, Justified: 395560, Finalized: 395559, Peers: 63",
    "17:05:51.527 INFO  - Slot Event  *** Slot: 12657927, Block: ... empty,    Justified: ...",
    "17:17:11.002 INFO  - Epoch Event *** Epoch: 395562, Justified checkpoint: 395561, Finalized checkpoint: 395560, Finalized root: e3032378ff11e040a689579c8a3eefa4e45485ee2ff270709a431106aac568a7",
    "18:23:07.855 WARN  - Late Block Import *** Block: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313) Proposer: 3126 Result: success Timings: arrival 8760ms, gossip_validation +0ms, pre-state_retrieved +2ms, processed +85ms, data_availability_checked +0ms, execution_payload_result_received +0ms, begin_importing +0ms, transaction_prepared +0ms, transaction_committed +0ms, completed +8ms",
    "18:23:13.482 INFO  - Reorg Event *** New Head: 6804d44b51fd3b957c147298575f53e8929769dfba4b6d80ca4725a08b50176d (12658314), Previous Head: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313), Common Ancestor: 41b24017534a5266d112fea39e7f7e10c1fade19272ee7e427b42cd586e2853a (12658312)"
)
val parsed = lines1.map(TekuLogParser::parse)

println(parsed.size)
parsed.forEach { println(it) }

6
ConfirmationRuleUpdate(headSlot=12657983, headShortRootHex=0x9acceb, confirmedSlot=12657982, confirmedShortRootHex=0x6dba83)
SlotEvent(time=17:17:03.812, slot=12657983, blockRootHex=9accebd69baa3e90411f123121f9eb7d2fc61f8117d81accf87f81c7bafde347)
SlotEvent(time=17:05:51.527, slot=12657927, blockRootHex=null)
Unknown(line=17:17:11.002 INFO  - Epoch Event *** Epoch: 395562, Justified checkpoint: 395561, Finalized checkpoint: 395560, Finalized root: e3032378ff11e040a689579c8a3eefa4e45485ee2ff270709a431106aac568a7)
Unknown(line=18:23:07.855 WARN  - Late Block Import *** Block: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313) Proposer: 3126 Result: success Timings: arrival 8760ms, gossip_validation +0ms, pre-state_retrieved +2ms, processed +85ms, data_availability_checked +0ms, execution_payload_result_received +0ms, begin_importing +0ms, transaction_prepared +0ms, transaction_committed +0ms, completed +8ms)
Unknown(line=18:23:13.482 INFO  - Reorg Event *** New

In [73]:
%use dataframe
import java.io.File

val logLines = File("./conf-logs-2.log").readLines()

data class ConfAndSlotEvents(
    val conf: LogEvent.ConfirmationRuleUpdate?,
    val slot: LogEvent.SlotEvent?,
)

val confirmLag = logLines
    .map { TekuLogParser.parse(it) }
    .scan(ConfAndSlotEvents(null, null), { scanAcc, event -> 
        when(event) {
            is LogEvent.ConfirmationRuleUpdate -> ConfAndSlotEvents(event, null)
            is LogEvent.SlotEvent -> scanAcc.copy(slot = event)
            else -> scanAcc.copy(slot = null)
        }
    })
    .filter { it.slot != null && it.conf != null }
    // leaving only events when confirmed slot advances 
    .zipWithNext()
    .mapNotNull { if (it.first.conf!!.confirmedSlot == it.second.conf!!.confirmedSlot) null else it.first }
    
    .map { it.slot!!.slot to (it.slot.slot - it.conf!!.confirmedSlot) }

val df = confirmLag.toDataFrame()
    .rename { all() }.into("slot", "confirm_lag")
    .inferType()
    
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
slot,Long,7502,7502,0,12657927,1,12662012.381498,2393.679287,12657927,12661969,12666150
confirm_lag,Long,7502,13,0,1,7098,1.142495,2.520961,1,1,160


In [3]:
%use kandy


In [77]:
import org.jetbrains.kotlinx.kandy.letsplot.layers.builders.subcontext.BorderLine

val df1 = df
    .filter { slot < 12_663_000 || slot > 12_663_200} // filter out disconnect and catch up sync 
    .filter { slot > 12657950 } // filter out initial sync delay 
//    .take(200)

df1.plot {
    x(slot) {
//        scale = continuous(
//            12_657_800L .. 12_666_300L 
//        )
    }
    y(confirm_lag){
        scale = continuous(
//            0L..10L,
//            transform = Transformation.LOG2,
        )
    }
    bars {
        this.borderLine {
            this.type = LineType.BLANK
            this.width = 0.0
        }
        fillColor = Color.BLUE
    }
    layout {
        size = 2500 to 500
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.3.3/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="v5ez8o"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"mapping":{
},
"data":{
"slot":[1.2657951E7,1.2657952E7,1.2657953E7,1.2657954E7,1.2657955E7,1.2657956E7,1.2657957E7,1.2657958E7,1.2657959E7,1.265796E7,1.2657961E7,1.2657962E7,1.2657963E7,1.2657964E7,1.2657965E7,1.2657966E7,1.2657967E7,1.2657968E7,1.2657969E7,1.265797E7,1.2657971E7,1.2657972E7,1.2657973E7,1.2657974E7,1.2657975E7,1.2657976E7,1.2657977E7,1.2657978E7,1.2657979E7,1.265798E7,1.2657981E7,1.2657982E7,1.2657983E7,1.2657984E7,1.2657985E7,1.2657986E7,1.2657987E7,1.2657988E7,1.2657989E7,1.265799E7,1.2657991E7,1.2657992E7,1.2657993E7,1.2657994E7,1.2657995E7,1.2657996E7,1.2657997E7,1.2657998E7,1.2657999E7,1.2658E7,1.2658001E7,1.2658002E7,1.2658003E7,1.2658004E7,1.2658005E7,1.2658006E7,1.2658007E7,1.2658008E7,1.2658009E7,1.265801E7,1.2658011E7,1.2658012E7,1.2658013E7,1.2658014E7,1.2658015E7,1.2658016E7,1.2658017E7,1.2658018E7,1.2658019E7,1.265802E7,1.2658021E7,1.2658022E7,1.2658023E7,1.2658024E7,1.2658025E7,1.2658028E7,1.2658029E7,1.265803E7,1.2658031E7,1.2658032E7,1.2658033E7,1.2658034E7,1.2658035E7,1.2658036E7,1.2658037E7,1.2658038E7,1.2658039E7,1.265804E7,1.2658041E7,1.2658042E7,1.2658043E7,1.2658044E7,1.2658045E7,1.2658046E7,1.2658047E7,1.2658049E7,1.265805E7,1.2658052E7,1.2658053E7,1.2658054E7,1.2658055E7,1.2658056E7,1.2658057E7,1.2658058E7,1.2658059E7,1.265806E7,1.2658061E7,1.2658062E7,1.2658063E7,1.2658064E7,1.2658065E7,1.2658066E7,1.2658067E7,1.2658068E7,1.2658069E7,1.265807E7,1.2658071E7,1.2658072E7,1.2658073E7,1.2658074E7,1.2658075E7,1.2658076E7,1.2658077E7,1.2658078E7,1.2658079E7,1.265808E7,1.2658081E7,1.2658082E7,1.2658083E7,1.2658084E7,1.2658085E7,1.2658086E7,1.2658087E7,1.2658088E7,1.2658093E7,1.2658094E7,1.2658095E7,1.2658096E7,1.2658097E7,1.2658098E7,1.2658099E7,1.26581E7,1.2658101E7,1.2658102E7,1.2658103E7,1.2658104E7,1.2658105E7,1.2658106E7,1.2658107E7,1.2658108E7,1.2658109E7,1.265811E7,1.2658111E7,1.2658112E7,1.2658113E7,1.2658114E7,1.2658115E7,1.2658116E7,1.2658117E7,1.2658118E7,1.2658119E7,1.265812E7,1.2658121E7,1.2658122E7,1.2658123E7,1.2658124E7,1.2658125E7,1.2658126E7,1.2658127E7,1.2658128E7,1.2658129E7,1.265813E7,1.2658131E7,1.2658133E7,1.2658134E7,1.2658135E7,1.2658137E7,1.2658138E7,1.2658139E7,1.265814E7,1.2658141E7,1.2658143E7,1.2658144E7,1.2658145E7,1.2658146E7,1.2658147E7,1.2658148E7,1.2658149E7,1.265815E7,1.2658151E7,1.2658152E7,1.2658153E7,1.2658154E7,1.2658155E7,1.2658156E7,1.2658157E7,1.2658158E7,1.2658159E7,1.265816E7,1.2658161E7,1.2658162E7,1.2658163E7,1.2658164E7,1.2658165E7,1.2658167E7,1.2658168E7,1.2658173E7,1.2658174E7,1.2658175E7,1.2658176E7,1.2658177E7,1.2658178E7,1.2658179E7,1.265818E7,1.2658181E7,1.2658182E7,1.2658183E7,1.2658184E7,1.2658185E7,1.2658186E7,1.2658187E7,1.2658188E7,1.2658189E7,1.265819E7,1.2658191E7,1.2658192E7,1.2658193E7,1.2658194E7,1.2658195E7,1.2658196E7,1.2658197E7,1.2658198E7,1.2658199E7,1.26582E7,1.2658201E7,1.2658202E7,1.2658203E7,1.2658204E7,1.2658205E7,1.2658206E7,1.2658207E7,1.2658208E7,1.2658209E7,1.265821E7,1.2658211E7,1.2658212E7,1.2658213E7,1.2658214E7,1.2658215E7,1.2658216E7,1.2658217E7,1.2658218E7,1.2658219E7,1.265822E7,1.2658221E7,1.2658222E7,1.2658223E7,1.2658224E7,1.2658225E7,1.2658226E7,1.2658227E7,1.2658228E7,1.2658229E7,1.265823E7,1.2658231E7,1.2658232E7,1.2658233E7,1.2658234E7,1.2658235E7,1.2658236E7,1.2658237E7,1.2658238E7,1.2658239E7,1.265824E7,1.2658241E7,1.2658242E7,1.2658244E7,1.2658245E7,1.2658246E7,1.2658247E7,1.2658248E7,1.2658249E7,1.265825E7,1.2658251E7,1.2658252E7,1.2658253E7,1.2658254E7,1.2658255E7,1.265826E7,1.2658261E7,1.2658262E7,1.2658263E7,1.2658264E7,1.2658265E7,1.2658266E7,1.2658267E7,1.2658268E7,1.2658269E7,1.265827E7,1.2658271E7,1.2658272E7,1.265827

In [69]:
import org.jetbrains.letsPlot.core.spec.back.transform.bistro.util.scale

plot { 
    histogram(df1.getColumn { confirm_lag }) {
        y { 
            scale = continuous(transform = Transformation.LOG2)
        }
        x {
//            scale = categorical(listOf(0,1,2,3,4,5,6,7,8))
        }
    } 
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.3.3/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="f3aC6D"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"confirm_lag",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null],
"trans":"LOG2"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"count":[7028.0,0.0,0.0,312.0,0.0,0.0,18.0,0.0,6.0,0.0,0.0,50.0,0.0,0.0,8.0,0.0,0.0,1.0,0.0,1.0],
"x":[1.0499999999999998,1.4,1.7499999999999998,2.0999999999999996,2.45,2.8,3.1499999999999995,3.5,3.8499999999999996,4.199999999999999,4.55,4.8999999999999995,5.25,5.6,5.949999999999999,6.3,6.6499999999999995,7.0,7.35,7.699999999999999]
},
"sampling":"none",
"position":"identity",
"geom":"bar"
}]
};
 var plotContainer = document.getElementById("f3aC6D");
 LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer);
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 64 
 
 
 
 
 
 
 256 
 
 
 
 
 
 
 1,024 
 
 
 
 
 
 
 4,096 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 confirm_lag